# Load Data

In [ ]:
import jax
import jax.numpy as jnp
import grain.python as grain
import tiktoken
from pathlib import Path

In [ ]:
from src.data_loader import load_text_from_file

from src.config import (
    TokenizerConfig,
    model_config,
    training_config,
    TINYSTORIES_FILE
)

## Sanity check content of the data set

In [ ]:
# Use config for file path
file_path = TINYSTORIES_FILE

with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
    data = f.read()
    stories = data.split(DELIMITER)

    print("first 300 characters:\n")
    print(stories[0][:300], "...")

    print(f"Total count of stories: {len(stories)}")

## Tokenizer

In [ ]:
# Define delimiter and tokenizer name explicitly for this data source
DELIMITER = "<|endoftext|>"
TOKENIZER_NAME = "gpt2"

# Create tokenizer config
tokenizer_config = TokenizerConfig(delimiter=DELIMITER, name=TOKENIZER_NAME)
tokenizer: tiktoken.Encoding = tokenizer_config.tokenizer

print(f"Vocabulary size of tokenizer: {tokenizer_config.vocab_size:,}")
print(f"Special tokens: {tokenizer.special_tokens_set}")

## Dataset Class

In [ ]:
class StoryDataset:
    
    def __init__(self, stories: list[str], maxlen: int, delimiter: str, tokenizer_name: str) -> None:
        self.stories: list[str] = stories
        self.maxlen: int = maxlen
        self.delimiter: str = delimiter
        self.tokenizer: tiktoken.Encoding = tiktoken.get_encoding(tokenizer_name)

        # constructs the token id for the delimiter so it can locate it in the future
        self.delimiter_token: int = self.tokenizer.encode(delimiter, \
                        allowed_special={delimiter})[0]

    def __len__(self) -> int:
        return len(self.stories)

    def __getitem__(self, idx: int) -> list[int]:
        story: str = self.stories[idx]

        # encode the entire story at that index (a string) into a list of token IDs using the tokenizer
        tokens: list[int] = self.tokenizer.encode(story, 
                                       allowed_special={self.delimiter})

        # truncate when the resulting tokens exceed the maximum set in the initializer
        if len(tokens) > self.maxlen:
            tokens: list[int] = tokens[:self.maxlen]

        # padding: add zeros to make sure all returned lists are the same size to input them all to the same tensors
        tokens.extend([0] * (self.maxlen - len(tokens)))
        
        return tokens

## Data Iterator (including shuffling with grain)

In [ ]:
shuffled_sampler = grain.IndexSampler(
    num_records=10,
    shuffle=True,
    seed=42,
    shard_options=grain.NoSharding(),    
    num_epochs=1
)

def print_sampler_example(sampler, name):
    print(f"\n{name}")
    for i, idx in enumerate(sampler):
        print(f"Record {i}: {idx}")

print_sampler_example(shuffled_sampler, "Shuffled sampler")

## Batch sequences into fixed-shape arrays

In [ ]:
batch_op_keep = grain.Batch(
    batch_size=32,
    drop_remainder=False # don't drop the remainder if if there is one
)

## Build a data loader

In [ ]:
def create_dataloader(
    stories: list[str],
    maxlen: int,
    batch_size: int,
    shuffle: bool = False, # defaults to no shuffle
    num_epochs: int = 1,
    seed: int = 42,
    worker_count: int = 0
 ) -> (grain.DataLoader, int):
    
    dataset: StoryDataset = StoryDataset(stories, maxlen, DELIMITER, TOKENIZER_NAME)
    estimated_batches: int = len(dataset) // batch_size

    # creates a new sampler
    sampler = grain.IndexSampler(
        num_records=len(dataset), # 1,000 stories for this dataset
        shuffle=shuffle,
        seed=seed,
        shard_options=grain.NoSharding(),
        num_epochs=num_epochs
    )

    # create new data loader
    dataloader = grain.DataLoader(
        data_source=dataset,
        sampler=sampler,

        # provide the operations that we want to perform on the dataset
        # in this case, the only operation being performed is batching
        operations=[
            grain.Batch(batch_size=batch_size, drop_remainder=True)
        ],
        
        worker_count=worker_count
    )
    
    return dataloader, estimated_batches

## Execute the code

In [ ]:
# load the stories using the explicit delimiter
paragraphs: list[str] = load_text_from_file(
    TINYSTORIES_FILE, 
    DELIMITER,
    max_paragraphs=training_config.max_stories
)

# sanity check the first paragraph
print(f"\n\n{paragraphs[0]}")

In [ ]:
# create the data loader, passing delimiter explicitly
dataloader, batches_per_epoch = create_dataloader(
    stories=paragraphs,
    maxlen=model_config.maxlen,
    batch_size=training_config.batch_size,
    delimiter=DELIMITER,
    shuffle=training_config.shuffle,
    num_epochs=1,
    seed=training_config.seed,
    worker_count=training_config.num_workers
)

print(f"\nDataLoader created successfully:")
print(f"Will produce {batches_per_epoch} batches per epoch")

## Fetch data

In [ ]:
# create iterator and call next on it
my_iterator = iter(dataloader)
next(my_iterator)